In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU Memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

PyTorch: 2.14.1+cu130
Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
GPU Memory: 6.0 GB


In [2]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)


block = DoubleConv(3, 32).to(DEVICE)

test_input = torch.randn(2, 3, 512, 512).to(DEVICE)

with torch.no_grad():
    test_output = block(test_input)

print("Input shape:", test_input.shape)
print("Output shape:", test_output.shape)
print("Output device:", test_output.device)

Input shape: torch.Size([2, 3, 512, 512])
Output shape: torch.Size([2, 32, 512, 512])
Output device: cuda:0


In [3]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.enc1 = DoubleConv(3, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)
        self.enc4 = DoubleConv(128, 256)

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.enc2(self.pool(x1))
        x3 = self.enc3(self.pool(x2))
        x4 = self.enc4(self.pool(x3))

        return x1, x2, x3, x4

In [5]:
encoder = Encoder().to(DEVICE)

test_input = torch.randn(2, 3, 512, 512).to(DEVICE)

with torch.no_grad():
    features = encoder(test_input)

print("Encoder output:", features[-1].shape)

Encoder output: torch.Size([2, 256, 64, 64])


In [6]:
class Bottleneck(nn.Module):
    def __init__(self):
        super().__init__()

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv = DoubleConv(256, 512)

    def forward(self, x):
        x = self.pool(x)
        x = self.conv(x)

        return x


bottleneck = Bottleneck().to(DEVICE)

with torch.no_grad():
    bottleneck_output = bottleneck(features[-1])

print("Bottleneck input:", features[-1].shape)
print("Bottleneck output:", bottleneck_output.shape)

Bottleneck input: torch.Size([2, 256, 64, 64])
Bottleneck output: torch.Size([2, 512, 32, 32])


In [7]:
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.up4 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(64, 32)

    def forward(self, x, features):
        x1, x2, x3, x4 = features

        x = self.up4(x)
        x = torch.cat([x, x4], dim=1)
        x = self.dec4(x)

        x = self.up3(x)
        x = torch.cat([x, x3], dim=1)
        x = self.dec3(x)

        x = self.up2(x)
        x = torch.cat([x, x2], dim=1)
        x = self.dec2(x)

        x = self.up1(x)
        x = torch.cat([x, x1], dim=1)
        x = self.dec1(x)

        return x

In [8]:
decoder = Decoder().to(DEVICE)

with torch.no_grad():
    decoder_output = decoder(bottleneck_output, features)

print("Bottleneck input:", bottleneck_output.shape)
print("Decoder output:", decoder_output.shape)
print("Output device:", decoder_output.device)

Bottleneck input: torch.Size([2, 512, 32, 32])
Decoder output: torch.Size([2, 32, 512, 512])
Output device: cuda:0


In [9]:
class OutputLayer(nn.Module):
    def __init__(self, in_channels=32, num_classes=40):
        super().__init__()

        self.conv = nn.Conv2d(
            in_channels,
            num_classes,
            kernel_size=1
        )

    def forward(self, x):
        return self.conv(x)


output_layer = OutputLayer().to(DEVICE)

with torch.no_grad():
    predictions = output_layer(decoder_output)

print("Decoder output:", decoder_output.shape)
print("Final output:", predictions.shape)

Decoder output: torch.Size([2, 32, 512, 512])
Final output: torch.Size([2, 40, 512, 512])


In [10]:
class UNet(nn.Module):
    def __init__(self, num_classes=40):
        super().__init__()

        self.encoder = Encoder()
        self.bottleneck = Bottleneck()
        self.decoder = Decoder()
        self.output_layer = OutputLayer(
            in_channels=32,
            num_classes=num_classes
        )

    def forward(self, x):
        features = self.encoder(x)

        bottleneck_output = self.bottleneck(features[-1])

        decoder_output = self.decoder(
            bottleneck_output,
            features
        )

        output = self.output_layer(decoder_output)

        return output


model = UNet(num_classes=40).to(DEVICE)

test_input = torch.randn(1, 3, 512, 512).to(DEVICE)

with torch.no_grad():
    test_output = model(test_input)

total_params = sum(p.numel() for p in model.parameters())

print("Input shape:", test_input.shape)
print("Output shape:", test_output.shape)
print("Total parameters:", f"{total_params:,}")
print("Model device:", next(model.parameters()).device)

Input shape: torch.Size([1, 3, 512, 512])
Output shape: torch.Size([1, 40, 512, 512])
Total parameters: 7,764,328
Model device: cuda:0


In [11]:
with torch.no_grad():
    predictions = model(test_input)

predicted_masks = torch.argmax(predictions, dim=1)

print("Model output:", predictions.shape)
print("Predicted mask:", predicted_masks.shape)
print("Predicted classes:", torch.unique(predicted_masks))

Model output: torch.Size([1, 40, 512, 512])
Predicted mask: torch.Size([1, 512, 512])
Predicted classes: tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 14, 15, 16, 17, 18,
        19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 31, 32, 33, 34, 35, 36, 37,
        38, 39], device='cuda:0')


In [12]:
semantic_logits = predictions[:, 1:, :, :]

semantic_masks = torch.argmax(
    semantic_logits,
    dim=1
) + 1

print("Semantic mask shape:", semantic_masks.shape)
print("Predicted semantic classes:", torch.unique(semantic_masks))

Semantic mask shape: torch.Size([1, 512, 512])
Predicted semantic classes: tensor([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 14, 15, 16, 17, 18, 19,
        20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 31, 32, 33, 34, 35, 36, 37, 38,
        39], device='cuda:0')
